In [1]:
from platform import python_version
print(python_version())

3.10.19


```
source image_venv_311/bin/activate
```

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [2]:
!nvidia-smi

Tue Nov 18 09:41:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   55C    P5              5W /   80W |    5485MiB /   6144MiB |     14%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:21:03_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0


In [4]:
import numpy as np
import os, sys
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from natsort import natsorted

In [10]:
import torch

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))

torch: 2.9.0+cu128
torch.version.cuda: 12.8
device: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [7]:
from cellpose import models, core, io, plot

#### run io.logger_setup() to get printing of progress

  - cellpose version: 	4.0.7 
  - platform:       	linux 
  - python version: 	3.11.14 
  - torch version:  	2.9.0+cu128

In [9]:
io.logger_setup()

2025-11-18 09:41:45,591 [INFO] WRITING LOG OUTPUT TO /home/flalix/.cellpose/run.log
2025-11-18 09:41:45,592 [INFO] 
cellpose version: 	4.0.7 
platform:       	linux 
python version: 	3.10.19 
torch version:  	2.9.0+cu128


(<Logger cellpose.io (INFO)>, PosixPath('/home/flalix/.cellpose/run.log'))

#### Check if colab notebook instance has GPU access

In [11]:
if core.use_gpu()==False:
    print(ImportError("No GPU access, change your runtime"))
else:
    print("GPU Ok")

2025-11-18 09:42:58,524 [INFO] ** TORCH CUDA version installed and working. **
GPU Ok


### Select an image

In [ ]:
is_laptop = True

if is_laptop:
    root_image = "../../colaboracoes/carlos_deOcesano"
else:
    root_image = "/media/flalix/d2f268d1-512d-499f-b3b3-6dad7d3fdd25/colaboracoes/deOcesano"
    
os.listdir(root_image)

In [ ]:
root_hcs = os.path.join(root_image, 'Plate1848')
hcs_folders = os.listdir(root_hcs)
print(root_hcs)
hcs_folders

In [ ]:
root_1perc = os.path.join(root_hcs, '1% SFB')
os.path.exists(root_1perc), root_1perc

In [ ]:
files = os.listdir(root_1perc)
for fname in files[:5]:
    print(fname)

In [ ]:
i=15
fname=files[i]
filefig = os.path.join(root_1perc, fname)
img = io.imread(filefig)

fig = plt.figure(figsize=(8, 8))
plt.imshow(img);

In [ ]:
print(f"\nImage '{fname}' in '{root_1perc}' has shape: {img.shape}. Assuming channel dimension is last with {img.shape[-1]} channels\n\n")

### Crop an image

box = (left, upper, right, lower)

In [ ]:
img.shape

In [ ]:
box = (0, 0, 248, 248)

In [ ]:
from PIL import Image

image = Image.open(filefig)

In [ ]:
seg_img = image.crop(box)

fig = plt.figure(figsize=(4, 4))
plt.imshow(seg_img);

In [ ]:
width, height = image.size
ncrop = 5

del_width = int(width/ncrop)
del_height = int(height/ncrop)

del_width, del_height

In [ ]:
np.zeros( (5,5) )

In [ ]:
from PIL import Image

image = Image.open(filefig)

fig = plt.figure(figsize=(8, 8))
plt.imshow(image);

In [ ]:
see_whole_image = True

if see_whole_image:
    fig = plt.figure(figsize=(8, 8))
    plt.imshow(img);

dic = {}

fig, axes = plt.subplots(ncrop, ncrop, figsize=(8, 8))

icount = -1
for ix in range(ncrop):
    xmin = ix*del_width

    if ix == ncrop-1:
        xmax = width
    else:
        xmax = xmin + del_width
    
    for iy in range(ncrop):
        ymin = iy*del_height
        
        dic[(ix,iy)] = [xmin, ymin]
        # print((ix,iy), [xmin, ymin])

        if iy == ncrop-1:
            ymax = height
        else:
            ymax = ymin + del_height

        icount +=1
        # box = (left, upper, right, lower)
        box = [xmin, ymin, xmax, ymax]
        seg_img = image.crop(box)

        # fig = plt.figure(figsize=(8, 8))
        # plt.imshow(seg_img)
        ax = axes[iy][ix]
        ax.imshow(seg_img)
        ax.xaxis.set_visible(False)
        ax.yaxis.set_visible(False)

plt.tight_layout()
plt.subplots_adjust(left=0.1,  # Left margin
                    right=0.9, # Right margin
                    bottom=0.1, # Bottom margin
                    top=0.9,   # Top margin
                    wspace=0.05, # Width space between subplots
                    hspace=0.05) # Height space between subplots